# AuditKit — one question, two correct tool routes

A reference (`Sample.expected_tool_calls`) holds one tool route. An agent can reach the same
correct answer a different way, and the alternative route used to score 0.0 on every
reference metric. `{"any_of": [route, ...]}` lists the routes that count, and each run is
scored against the route it actually took.

| Part | What you see |
|---|---|
| 1 | The same two runs under a one-route reference, then under `any_of` |
| 2 | Both routes are correct, and the score says which one the run took |
| 3 | All three reference metrics read the **same** route |
| 4 | The tie between routes is visible, not silent |
| 5 | `[]` as a route: answering without a tool can also be right |
| 6 | A run that mixes two routes is scored against one whole route |
| 7 | A reference that cannot be read raises, loudly |
| 8 | A plain reference is unchanged |

Everything runs offline on CPU: `model="precomputed"` with a recorded trace — no API key, no
server, no GPU. `any_of` is a reference-format feature, so there is nothing for a model to do.

## 1. The problem: one answer, two routes

A predicted call only matches a reference call with the same tool name, so naming one route
is really a claim that the agent should have taken that route. "Which is warmer, Paris or
Rome?" has two correct answers: look the two cities up independently, or ask for both in one
`compare_weather` call. Both runs below produced the same right answer.

In [ ]:
import auditkit as ak
from auditkit.metrics.agent import MAX_REFERENCE_PATHS
from auditkit.trace import to_reference_paths, to_turns


def call(name, **arguments):
    return {"name": name, "arguments": arguments}


TOOLS = [{"type": "function", "function": {
              "name": "get_weather", "description": "Current weather for one city.",
              "parameters": {"type": "object", "properties": {"city": {"type": "string"}},
                             "required": ["city"]}}},
         {"type": "function", "function": {
              "name": "compare_weather", "description": "Current weather for two or more cities at once.",
              "parameters": {"type": "object",
                             "properties": {"cities": {"type": "array", "items": {"type": "string"}}},
                             "required": ["cities"]}}}]

QUESTION = "Which is warmer right now, Paris or Rome?"
ANSWER = "Rome is warmer: 23C against 14C."
REFERENCE_SCORERS = [ak.ToolCallF1(), ak.TrajectoryMatch(), ak.ParallelToolCalls()]

PARIS, ROME = call("get_weather", city="Paris"), call("get_weather", city="Rome")
COMPARE = call("compare_weather", cities=["Paris", "Rome"])

# A route is the reference format used everywhere else in AuditKit: a list of turns, each
# turn the calls the agent makes in one response.
TWO_LOOKUPS, ONE_LOOKUP = [[PARIS, ROME]], [[COMPARE]]


def score_run(reference, turns):
    """Score one recorded run (`turns`) against one reference. No model, no server."""
    sample = ak.Sample(input=QUESTION, tools=TOOLS, expected_tool_calls=reference,
                       actual_output=ANSWER, actual_trace={"tool_calls": turns})
    return ak.evaluate([sample], model="precomputed", scorers=REFERENCE_SCORERS)


def score_docs(result):
    """Every score the Runner recorded, keyed by name, with its metadata."""
    return {d["name"]: d for d in result.predictions[0].metadata["scores"]}

In [ ]:
# Each reference below names ONE route, so the run that took the other one scores 0.0.
for reference, label in ((ONE_LOOKUP, "reference: one compare_weather call"),
                         (TWO_LOOKUPS, "reference: two get_weather calls")):
    print(label)
    for run, turns in (("run: two get_weather", TWO_LOOKUPS), ("run: one compare_weather", ONE_LOOKUP)):
        d = score_docs(score_run(reference, turns))
        print(f"   {run:<24} f1={d['tool_call_f1']['value']:.2f}  "
              f"trajectory_strict={d['trajectory_strict']['value']:.2f}  "
              f"parallel_detection={d['parallel_detection']['value']:.2f}")

The off-diagonal is the bug, and it is in the dataset rather than the metric: a run that
answered correctly scored 0.0 because the author wrote down the other route.
`parallel_tool_calls` reads the same reference, which is why all three move together.

The fix is the reference. `{"any_of": [...]}` names every route that is correct, each route
in the format above.

In [ ]:
EITHER_ROUTE = {"any_of": [TWO_LOOKUPS, ONE_LOOKUP]}   # every listed route is a correct answer

for run, turns in (("two get_weather", TWO_LOOKUPS), ("one compare_weather", ONE_LOOKUP)):
    f1 = score_docs(score_run(EITHER_ROUTE, turns))["tool_call_f1"]
    m = f1["metadata"]
    print(f"{run:<20} f1={f1['value']:.2f}  matched_path={m['matched_path']}  "
          f"n_paths={m['n_paths']}  path_f1={m['path_f1']}")
    assert f1["value"] == 1.0 and m["matched_path"] == m["path_f1"].index(1.0)

## 2. Both routes are correct

`f1` is 1.0 either way, and `matched_path` says which route the run was read against. The
scores carry:

| Key | Meaning |
|---|---|
| `matched_path` | index of the route this run was scored against |
| `n_paths` | how many routes the sample listed |
| `path_f1` | F1 against every route, in the listed order, so the choice can be checked |

`auditkit.trace.to_reference_paths` normalises both reference forms to the same thing — a
list of routes — and never expands or combines them: routes are alternatives, not a tree.

In [ ]:
print("plain reference ->", len(to_reference_paths(TWO_LOOKUPS)), "route,  any_of ->",
      len(to_reference_paths(EITHER_ROUTE)), "routes")

PATH_KEYS = ("matched_path", "n_paths", "path_f1")


def scores_only(result):
    """(value, metadata) per score with the route bookkeeping dropped: what is left has to
    match, key for key, against the same route written on its own."""
    return {name: (d["value"], {k: v for k, v in d.get("metadata", {}).items() if k not in PATH_KEYS})
            for name, d in score_docs(result).items()}


# Listing a second, wrong route cannot move a perfect run off what its own route said.
print("route 0 alone == route 0 inside any_of:",
      scores_only(score_run(TWO_LOOKUPS, TWO_LOOKUPS)) == scores_only(score_run(EITHER_ROUTE, TWO_LOOKUPS)))

## 3. All three reference metrics read the same route

One route is chosen per run and `tool_call_f1`, `trajectory_match` and `parallel_tool_calls`
all score *that* route. Otherwise `tool_call_f1` could take its recall from route 0 while
`parallel_tool_calls` takes its parallelism from route 1, and three columns of a report would
describe two different runs.

The route is reported on every score of the family, not only the first one: a `matched_path`
on `tool_call_f1` and nothing on `trajectory_strict` would be exactly the bug this prevents.

In [ ]:
for run, turns in (("two get_weather", TWO_LOOKUPS), ("one compare_weather", ONE_LOOKUP)):
    docs = score_docs(score_run(EITHER_ROUTE, turns))
    print(f"\n{run}")
    for name, d in docs.items():
        m = d["metadata"]
        print(f"   {name:<22}{d['value']:>5.2f}   matched_path={m['matched_path']}  "
              f"n_paths={m['n_paths']}  path_f1={m['path_f1']}")
    # Every score names the same route, so the three metrics describe one run.
    assert {d["metadata"]["matched_path"] for d in docs.values()} == \
           {docs["tool_call_f1"]["metadata"]["matched_path"]}

## 4. The tie is visible

Routes are ranked by call-level F1 over the flattened calls, so the choice ignores turn
structure and no metric can pick a different winner. Ties break on higher recall — a partial
run is credited with the route it got furthest along — and only then on the order the routes
are listed. `path_f1` shows what every route scored, so a reader who disagrees with the
choice can see exactly what it was made from.

In [ ]:
# This run matched 2 of the 6 calls the long route expects and 1 of the 2 the short one does,
# so F1 = 2 * matched / (predicted + expected) is 0.5 against either route. Only the recall
# tiebreak can separate them -- and it picks the short route whatever the listing order.
ELSEWHERE = [call("get_weather", city=city) for city in ("Berlin", "Oslo", "Madrid", "Lisbon")]
SHORT, LONG = [[PARIS, ROME]], [[PARIS, call("get_weather", city="Berlin")] + ELSEWHERE]
HALF_DONE = [[PARIS, call("get_weather", city="Berlin")]]

for listed, order in (("short, long", [SHORT, LONG]), ("long, short", [LONG, SHORT])):
    docs = score_docs(score_run({"any_of": order}, HALF_DONE))
    f1, recall = docs["tool_call_f1"], docs["tool_call_recall"]
    print(f"listed {listed:<12} path_f1={f1['metadata']['path_f1']}  "
          f"matched_path={f1['metadata']['matched_path']}  that route's recall={recall['value']:.3f}")

In [ ]:
# A complete tie: F1 and recall are equal, so the listed order decides. The choice is visible
# because it changes what trajectory_strict says -- the two routes are different trajectories.
SPLIT, BATCHED = [[PARIS], [ROME]], [[PARIS, ROME]]

for label, order, first in (("split first", [SPLIT, BATCHED], "split"),
                            ("batched first", [BATCHED, SPLIT], "batched")):
    docs = score_docs(score_run({"any_of": order}, SPLIT))
    print(f"route 0 = {first:<8} matched_path={docs['tool_call_f1']['metadata']['matched_path']}  "
          f"path_f1={docs['tool_call_f1']['metadata']['path_f1']}  "
          f"trajectory_strict={docs['trajectory_strict']['value']:.2f}")

## 5. `[]` is a route

The empty list is a route, meaning "calling no tool is also correct". That generalises the
existing irrelevance case (`expected_tool_calls=[]`) rather than replacing it: on an ambiguous
request, answering with what you have and asking which one is meant are both right, while an
unrelated third call is wrong against both routes.

In [ ]:
ASK = call("ask_clarification", question="Which Paris booking?")
ASK_OR_ANSWER = {"any_of": [[], [[ASK]]]}    # answer with no tool, or ask which one
print("calls per route:", [sum(len(turn) for turn in route)
                            for route in to_reference_paths(ASK_OR_ANSWER)])

for run, turns in (("answered, no tool", []), ("asked for clarification", [[ASK]]),
                   ("did something else", [[call("book_flight", flight_id="AZ-61")]])):
    f1 = score_docs(score_run(ASK_OR_ANSWER, turns))["tool_call_f1"]
    print(f"{run:<26} f1={f1['value']:.2f}  matched_path={f1['metadata']['matched_path']}  "
          f"path_f1={f1['metadata']['path_f1']}")

## 6. A mixed run is scored against one route, never both

A run is one trajectory, so it is measured against the single route it got furthest along —
not the best part of every route. This run took half of each: `get_weather(Paris)` from
route 0, and `compare_weather` from route 1, which is one call plus a call it did not ask
for. Crediting the best part of every route would read precision 1.0 and recall 1.0.

In [ ]:
docs = score_docs(score_run(EITHER_ROUTE, [[PARIS, COMPARE]]))
print({name: round(d["value"], 3) for name, d in docs.items() if name.startswith("tool_call")})

counted = docs["tool_call_precision"]["metadata"]
print(f"route {docs['tool_call_f1']['metadata']['matched_path']} expects {counted['n_expected']} call; "
      f"the run made {counted['n_predicted']} and matched {counted['n_matched']}, "
      f"path_f1={docs['tool_call_f1']['metadata']['path_f1']}")

## 7. A reference that cannot be read raises

A reference that cannot be parsed is a wrong score waiting to happen, and a key silently
ignored beside `any_of` is exactly that. These are `ValueError`s naming the problem, so a bad
dataset cell is reported rather than scored.

In [ ]:
for bad in ({"any_of": []},                                   # no routes at all
            {"any_of": [TWO_LOOKUPS], "note": "..."}):        # a key beside any_of
    try:
        to_reference_paths(bad)
    except ValueError as e:
        print("ValueError:", e, "\n")

# Past the route budget the same rule applies at scoring time, with the advice to split the
# sample rather than a slow, silent run.
try:
    too_wide = {"any_of": [[[PARIS]]] * (MAX_REFERENCE_PATHS + 1)}
    ak.ToolCallF1().score(ak.Sample(input=QUESTION, expected_tool_calls=too_wide), "",
                          {"trace": {"tool_calls": TWO_LOOKUPS}})
except ValueError as e:
    print("ValueError:", e)

## 8. A plain reference is unchanged

Every reference shape that used to be valid is still exactly one route, and a single-route
reference gets no path metadata at all: `matched_path: 0` on every existing score would
change the metadata of every existing run and every saved report, for nothing. The two forms
cannot collide either — `to_turns` still refuses `{"any_of": ...}`, because its value is a
list and no call shape reads a list as a call.

In [ ]:
plain = score_run(TWO_LOOKUPS, TWO_LOOKUPS)
keys = {k for d in score_docs(plain).values() for k in d.get("metadata", {}) if k in PATH_KEYS}
print("a plain reference is one route:", to_reference_paths(TWO_LOOKUPS) == [to_turns(TWO_LOOKUPS)],
      "| path keys on its scores:", keys or "none")

try:
    to_turns(EITHER_ROUTE)
except ValueError as e:
    print(f"to_turns({{'any_of': ...}}) -> ValueError: {str(e)[:60]}...")

---

## What this does not do yet

This is whole-route alternatives and nothing more. Deliberately not here:

| Deferred | What it would take |
|---|---|
| `{"one_of": [...]}` per-call slots | `search_web` and `lookup_kb` as the same step inside one turn. It is also the fix for the combinatorial case: three independent two-way choices need 8 listed routes today |
| A dependency graph (`depends_on`) | Calls declare what they need, and any schedule respecting it is valid. It would fold parallel-vs-serial and many route variants into one definition, but it earns its keep only if whole-route alternatives fall short on real data |
| The BFCL `possible_answer` loader | The format this exists for: alternative answers → `any_of`, alternative function names → `one_of`, alternative argument values → `arg_match`. It needs the real file in hand, which is why `one_of` is waiting on it |
| A route distribution in the report | How many runs took each route. `matched_path` is on every score, so the data is there; the aggregation is report work and has no better/worse direction |

Two limits worth knowing: one sample may list at most `MAX_REFERENCE_PATHS = 64` routes
(each costs a maximum matching when the route is chosen), and every listed route is equally
correct — how efficient a route was is `parallel_tool_calls` and `redundant_tool_calls`,
which measure the run itself.

`AgentCase.reference_turns`, the `agent_eval` front door, takes the same forms and reports the
same keys, so a stored agent-eval report and a fresh `ak.evaluate` run describe the same run.